In [1]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
import warnings

warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)

In [2]:
from core.backtesting import BacktestingEngine

backtesting = BacktestingEngine(root_path=root_path, load_cached_data=False)

In [3]:
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
from controllers.directional_trading.elitesmugplugobi import EliteSmugPlugObiConfig  # Fixed import
import datetime
from decimal import Decimal


# Controller configuration
connector_name = "binance"
trading_pair = "WLD-USDT"
interval = "1m"
total_amount_quote = 1000
max_executors_per_side = 2
take_profit = 0.29
stop_loss = 0.06
trailing_stop_activation_price = 0.025
trailing_stop_trailing_delta = 0.019
time_limit = 60 * 60 * 8  # 8 hours
cooldown_time = 60  # 1 minute

# EliteSmugPlugObi specific parameters
ema_fast = 8  # Range: 3-8
ema_medium = 21  # Range: 8-21
ema_slow = 34  # Range: 13-34
atr_length = 10  # Range: 10-21
atr_multiplier = 0.9  # Range: 0.8-1.5
volume_surge = 3.1  # Range: 1.5-3.0
rsi_period = 13  # Range: 8-21
obi_window = 20  # Range: 10-30
obi_threshold = 0.6  # Range: 0.5-0.8
bid_ask_ratio_threshold = 1.2  # Range: 1.1-1.5


# Creating the instance of the configuration and the controller
config = EliteSmugPlugObiConfig(
    id=f"elitesmugplugobi_{connector_name}_{interval}_{trading_pair}",
    connector_name=connector_name,
    trading_pair=trading_pair,
    interval=interval,
    ema_fast=ema_fast,
    ema_medium=ema_medium,
    ema_slow=ema_slow,
    atr_length=atr_length,
    atr_multiplier=atr_multiplier,
    volume_surge=Decimal(volume_surge),
    rsi_period=rsi_period,
    obi_window=20,  # Added OBI specific parameters
    obi_threshold=0.6,
    bid_ask_ratio_threshold=1.2,
    total_amount_quote=Decimal(total_amount_quote),
    take_profit=Decimal(take_profit),
    stop_loss=Decimal(stop_loss),
    trailing_stop=TrailingStop(
        activation_price=Decimal(trailing_stop_activation_price),
        trailing_delta=Decimal(trailing_stop_trailing_delta)
    ),
    time_limit=time_limit,
    max_executors_per_side=max_executors_per_side,
    cooldown_time=cooldown_time,
)

In [4]:
# Running the backtesting this will output a backtesting result object that has built in methods to visualize the results

start = int(datetime.datetime(2025, 1, 1).timestamp())
end = int(datetime.datetime(2025, 1, 30).timestamp())


backtesting_result = await backtesting.run_backtesting(config, start, end, "1s")

2025-02-08 02:14:00,367 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x17963bac0>
2025-02-08 02:14:00,416 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x17961bdc0>, 701075.347874875)])']
connector: <aiohttp.connector.TCPConnector object at 0x17963b880>
2025-02-08 02:14:22,463 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x17a17d9c0>
2025-02-08 02:14:22,474 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x17a1638e0>, 701097.44554225)])']
connector: <aiohttp.connector.TCPConnector object at 0x17a17dea0>


In [5]:
# ... existing imports and setup ...

def format_backtesting_results(backtesting_result):
    """Format backtesting results for display"""
    results = backtesting_result.results
    
    try:
        net_pnl_quote = results.get("net_pnl_quote", 0)
        net_pnl_pct = results.get("net_pnl", 0)
        max_drawdown = results.get("max_drawdown_usd", 0)
        max_drawdown_pct = results.get("max_drawdown_pct", 0)
        total_volume = results.get("total_volume", 0)
        sharpe_ratio = results.get("sharpe_ratio", 0)
        profit_factor = results.get("profit_factor", 0)
        total_executors = results.get("total_executors", 0)
        accuracy_long = results.get("accuracy_long", 0)
        accuracy_short = results.get("accuracy_short", 0)
        
        # Handle close_types safely
        close_types = results.get("close_types", {})
        if isinstance(close_types, dict):
            take_profit = close_types.get("TAKE_PROFIT", 0)
            stop_loss = close_types.get("STOP_LOSS", 0)
            time_limit = close_types.get("TIME_LIMIT", 0)
            trailing_stop = close_types.get("TRAILING_STOP", 0)
            early_stop = close_types.get("EARLY_STOP", 0)
        else:
            take_profit = stop_loss = time_limit = trailing_stop = early_stop = 0
        
        return f"""
Net PNL: ${net_pnl_quote:.2f} ({net_pnl_pct*100:.2f}%) | Max Drawdown: ${max_drawdown:.2f} ({max_drawdown_pct*100:.2f}%)
Total Volume ($): {total_volume:.2f} | Sharpe Ratio: {sharpe_ratio:.2f} | Profit Factor: {profit_factor:.2f}
Total Executors: {total_executors} | Accuracy Long: {accuracy_long:.2f} | Accuracy Short: {accuracy_short:.2f}
Close Types: Take Profit: {take_profit} | Stop Loss: {stop_loss} | Time Limit: {time_limit} |
             Trailing Stop: {trailing_stop} | Early Stop: {early_stop}
"""
    except Exception as e:
        return f"Error formatting results: {str(e)}\nRaw results: {results}"

# Run backtesting
backtesting_result = await backtesting.run_backtesting(config, start, end, "1m")

# Display formatted results
print(format_backtesting_results(backtesting_result))

# Display visualization
backtesting_result.get_backtesting_figure()


Net PNL: $0.00 (0.00%) | Max Drawdown: $0.00 (0.00%)
Total Volume ($): 0.00 | Sharpe Ratio: 0.00 | Profit Factor: 0.00
Total Executors: 0 | Accuracy Long: 0.00 | Accuracy Short: 0.00
Close Types: Take Profit: 0 | Stop Loss: 0 | Time Limit: 0 |
             Trailing Stop: 0 | Early Stop: 0



In [6]:
# Let's see what is inside the backtesting results
print(backtesting_result.get_results_summary())
backtesting_result.get_backtesting_figure()

AttributeError: 'int' object has no attribute 'get'

In [ ]:
# 2. The executors dataframe: this is the dataframe that contains the information of the orders that were executed
import pandas as pd

executors_df = backtesting_result.executors_df
executors_df.head()

### Backtesting Analysis

### Scatter of PNL per Trade
This bar chart illustrates the PNL for each individual trade. Positive PNLs are shown in green and negative PNLs in red, providing a clear view of profitable vs. unprofitable trades.


In [ ]:
import plotly.express as px

# Create a new column for profitability
executors_df['profitable'] = executors_df['net_pnl_quote'] > 0

# Create the scatter plot
fig = px.scatter(
    executors_df,
    x="timestamp",
    y='net_pnl_quote',
    title='PNL per Trade',
    color='profitable',
    color_discrete_map={True: 'green', False: 'red'},
    labels={'timestamp': 'Timestamp', 'net_pnl_quote': 'Net PNL (Quote)'},
    hover_data=['filled_amount_quote', 'side']
)

# Customize the layout
fig.update_layout(
    xaxis_title="Timestamp",
    yaxis_title="Net PNL (Quote)",
    legend_title="Profitable",
    font=dict(size=12, color="white"),
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0.8)',  # Dark background
    paper_bgcolor='rgba(0,0,0,0.8)',  # Dark background for the entire plot area
    xaxis=dict(gridcolor="gray"),
    yaxis=dict(gridcolor="gray")
)

# Add a horizontal line at y=0 to clearly separate profits and losses
fig.add_hline(y=0, line_dash="dash", line_color="lightgray")

# Show the plot
fig.show()

### Histogram of PNL Distribution
The histogram displays the distribution of PNL values across all trades. It helps in understanding the frequency and range of profit and loss outcomes.


In [ ]:
fig = px.histogram(executors_df, x='net_pnl_quote', title='PNL Distribution')
fig.show()


# Conclusion
We can see that the indicator has potential to bring good signals to trade and might be interesting to see how we can design a market maker that shifts the mid price based on this indicator.
A lot of the short signals are wrong but if we zoom in into the loss signals we can see that the losses are not that big and the wins are bigger and if we had implemented the trailing stop feature probably a lot of them are going to be profits.

# Next steps
- Filter only the loss signals and understand what you can do to prevent them
- Try different configuration values for the indicator
- Test in multiple markets, pick mature markets like BTC-USDT or ETH-USDT and also volatile markets like DOGE-USDT or SHIB-USDT